# 🤖 Smart Assistant with Code Execution — LangChain

**قابلیت‌ها:**
- تشخیص اینکه سوال نیاز به محاسبات ریاضی داره یا نه
- تولید کد پایتون و اجرای خودکار اون
- retry خودکار در صورت خطا (تا ۳ بار)
- تولید پاسخ نهایی انسانی و طبیعی

این نسخه با **LangChain** و `init_chat_model` بازنویسی شده.

In [2]:
import io
import sys

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

In [3]:
load_dotenv()

# برای تغییر مدل یا provider فقط همین دو خط را عوض کنید:
# init_chat_model("claude-haiku-4-5", model_provider="anthropic")
# init_chat_model("qwen3",            model_provider="ollama")
MODEL      = "gpt-4o-mini"
PROVIDER   = "openai"

In [ ]:
from pydantic import BaseModel, Field


class MathCheck(BaseModel):
    """تشخیص اینکه آیا سوال نیاز به محاسبات ریاضی دارد."""
    needs_math: bool = Field(
        description="True اگر سوال نیاز به محاسبات ریاضی یا اجرای کد پایتون داشته باشد، در غیر این صورت False."
    )
    reasoning: str = Field(
        description="توضیح کوتاه دلیل تصمیم"
    )


def check_if_needs_math(prompt: str) -> bool:
    """از مدل می‌پرسد آیا این سوال نیاز به محاسبات ریاضی دارد یا نه.
    از with_structured_output و Pydantic برای خروجی type-safe استفاده می‌کند.
    """

    llm = init_chat_model(MODEL, model_provider=PROVIDER, temperature=0)
    structured_llm = llm.with_structured_output(MathCheck)

    messages = [
        SystemMessage(content="تو یک دستیار هوشمند هستی که تشخیص می‌دهی آیا یک سوال نیاز به محاسبات ریاضی یا برنامه‌نویسی دارد."),
        HumanMessage(content=f"آیا سوال زیر نیاز به محاسبات ریاضی یا اجرای کد پایتون دارد؟\n\nسوال: {prompt}")
    ]

    result: MathCheck = structured_llm.invoke(messages)
    print(f"   🧠 reasoning: {result.reasoning}")
    return result.needs_math


def execute_code_safely(code: str):
    """کد پایتون را به‌صورت ایمن اجرا می‌کند و خروجی را برمی‌گرداند."""
    old_stdout = sys.stdout
    sys.stdout = buffer = io.StringIO()

    # حذف markdown fences از کد
    clean_code = code.strip()
    for marker in ["```python", "```Python", "```", "python"]:
        clean_code = clean_code.replace(marker, "")
    clean_code = clean_code.strip()

    error = None
    try:
        exec(clean_code, {})
    except Exception as e:
        error = str(e)
    finally:
        sys.stdout = old_stdout

    output = buffer.getvalue()
    return output, error, clean_code


def generate_code(prompt: str, error_context: str = None, broken_code: str = None) -> str:
    """از مدل می‌خواهد کد پایتون تولید کند (یا خطا را برطرف کند)."""
    llm = init_chat_model(MODEL, model_provider=PROVIDER, temperature=0.2)

    if error_context and broken_code:
        user_content = (
            f"The following code has this error: {error_context}\n\n"
            f"Code:\n{broken_code}\n\n"
            f"Main question: {prompt}\n\n"
            "Fix the code."
        )
    else:
        user_content = (
            prompt
            + "\n\nWrite only Python code. Do not provide any explanation. Use print() to display the result."
        )

    messages = [
        SystemMessage(content=(
            "You are a Python programmer. "
            "Write only pure Python code, without any explanation or markdown. "
            "Always use print() to display the result."
        )),
        HumanMessage(content=user_content)
    ]

    response = llm.invoke(messages)
    return response.content


def generate_human_response(original_question: str, code: str, output: str) -> str:
    """بر اساس خروجی کد، یک پاسخ طبیعی و انسانی تولید می‌کند."""
    llm = init_chat_model(MODEL, model_provider=PROVIDER, temperature=0.7)

    messages = [
        SystemMessage(content=(
            "You are an intelligent assistant who must respond naturally and humanly, "
            "without showing code or technical details."
        )),
        HumanMessage(content=(
            f"User question: {original_question}\n\n"
            f"Calculation result: {output}\n\n"
            "Based on the result, provide a complete, clear, and human-like response. "
            "Only give the final answer."
        ))
    ]

    response = llm.invoke(messages)
    return response.content


def generate_text_response(prompt: str) -> str:
    """پاسخ معمولی بدون نیاز به کد."""
    llm = init_chat_model(MODEL, model_provider=PROVIDER, temperature=0.7)

    messages = [
        SystemMessage(content="You are an intelligent and helpful assistant."),
        HumanMessage(content=prompt)
    ]

    response = llm.invoke(messages)
    return response.content


print("✅ تمام توابع آماده‌ست!")

✅ تمام توابع آماده‌ست!


In [5]:
def smart_assistant(prompt: str, max_attempts: int = 3, verbose: bool = True) -> dict:
    """
    دستیار هوشمند که:
    - تشخیص میده سوال ریاضیه یا نه
    - در صورت نیاز کد تولید و اجرا میکنه
    - retry خودکار در صورت خطا
    - پاسخ انسانی تولید میکنه

    Args:
        prompt: سوال کاربر
        max_attempts: حداکثر تلاش برای fix کردن کد (پیش‌فرض: 3)
        verbose: نمایش جزئیات اجرا (پیش‌فرض: True)

    Returns:
        dict: شامل answer، code (اگه بود)، output کد، و type
    """
    print(f"\n{'='*60}")
    print(f"❓ سوال: {prompt}")
    print('='*60)

    # مرحله ۱: تشخیص نوع سوال
    print("🔍 در حال بررسی نوع سوال...")
    needs_math = check_if_needs_math(prompt)

    if needs_math:
        print("🔢 این سوال نیاز به محاسبات ریاضی دارد.")

        error_context = None
        broken_code = None

        for attempt in range(1, max_attempts + 1):
            print(f"\n⚙️  تلاش {attempt} از {max_attempts}...")

            # تولید کد
            raw_code = generate_code(prompt, error_context, broken_code)

            # اجرای کد
            output, error, clean_code = execute_code_safely(raw_code)

            if verbose:
                print(f"📝 کد تولید شده:\n{'-'*40}\n{clean_code}\n{'-'*40}")

            if error:
                print(f"⚠️  خطا: {error}")
                error_context = error
                broken_code = clean_code
            else:
                print(f"✅ محاسبات با موفقیت انجام شد!")
                print(f"📊 خروجی کد: {output.strip()}")

                # تولید پاسخ انسانی
                print("\n💬 در حال تولید پاسخ نهایی...")
                human_answer = generate_human_response(prompt, clean_code, output)

                print(f"\n{'='*60}")
                print("💬 پاسخ نهایی:")
                print(human_answer)
                print('='*60)

                return {
                    "answer": human_answer,
                    "code": clean_code,
                    "output": output,
                    "attempts": attempt,
                    "type": "math"
                }

        print("❌ متاسفانه پس از چند تلاش نتوانستیم مسئله را حل کنیم.")
        return {"answer": None, "error": "Max attempts reached", "type": "math"}

    else:
        # سوال معمولی
        print("💬 در حال پاسخ دادن...")
        answer = generate_text_response(prompt)

        print(f"\n{'='*60}")
        print("💬 پاسخ:")
        print(answer)
        print('='*60)

        return {
            "answer": answer,
            "code": None,
            "output": None,
            "type": "text"
        }


print("✅ تابع smart_assistant آماده‌ست!")

✅ تابع smart_assistant آماده‌ست!


In [6]:
result = smart_assistant("ریشه دوم ۱۴۴ چقدر است؟")


❓ سوال: ریشه دوم ۱۴۴ چقدر است؟
🔍 در حال بررسی نوع سوال...
   🧠 reasoning: این سوال نیاز به محاسبات ریاضی دارد زیرا باید ریشه دوم عدد ۱۴۴ محاسبه شود.
🔢 این سوال نیاز به محاسبات ریاضی دارد.

⚙️  تلاش 1 از 3...
📝 کد تولید شده:
----------------------------------------
print(144**0.5)
----------------------------------------
✅ محاسبات با موفقیت انجام شد!
📊 خروجی کد: 12.0

💬 در حال تولید پاسخ نهایی...

💬 پاسخ نهایی:
ریشه دوم ۱۴۴ برابر با ۱۲ است.


In [7]:
result = smart_assistant("اگه ۱۵ درصد تخفیف روی قیمت ۲۵۰,۰۰۰ تومان بخوریم، قیمت نهایی چقدر میشه؟")


❓ سوال: اگه ۱۵ درصد تخفیف روی قیمت ۲۵۰,۰۰۰ تومان بخوریم، قیمت نهایی چقدر میشه؟
🔍 در حال بررسی نوع سوال...
   🧠 reasoning: برای محاسبه قیمت نهایی بعد از اعمال تخفیف، باید ۱۵ درصد از ۲۵۰,۰۰۰ تومان را محاسبه کنیم و سپس آن را از قیمت اولیه کم کنیم.
🔢 این سوال نیاز به محاسبات ریاضی دارد.

⚙️  تلاش 1 از 3...
📝 کد تولید شده:
----------------------------------------
price = 250000
discount = 0.15
final_price = price * (1 - discount)
print(final_price)
----------------------------------------
✅ محاسبات با موفقیت انجام شد!
📊 خروجی کد: 212500.0

💬 در حال تولید پاسخ نهایی...

💬 پاسخ نهایی:
قیمت نهایی پس از اعمال ۱۵ درصد تخفیف روی ۲۵۰,۰۰۰ تومان، ۲۱۲,۵۰۰ تومان می‌شود.


In [8]:
result = smart_assistant("پایتخت فرانسه کجاست؟")


❓ سوال: پایتخت فرانسه کجاست؟
🔍 در حال بررسی نوع سوال...
   🧠 reasoning: این سوال به اطلاعات عمومی مربوط می‌شود و نیازی به محاسبات ریاضی یا اجرای کد پایتون ندارد.
💬 در حال پاسخ دادن...

💬 پاسخ:
پایتخت فرانسه پاریس است. پاریس یکی از شهرهای معروف و تاریخی جهان به شمار می‌رود و به خاطر جاذبه‌های فرهنگی و تاریخی‌اش شناخته می‌شود.
